# 01 · Carga y limpieza de datos

**Proyecto semestral IMT2200, Grupo 6:** *¿Mide el NEM lo mismo para todos?*

Este notebook carga las 16 tablas del MINEDUC (4 tipos × procesos 2023-2026),
documenta cada decisión de limpieza y guarda el panel unificado en
`data/processed/panel_completo.parquet`, que es la entrada de `02_eda.ipynb`.
Las funciones están en `src/` y los parámetros en `src/config.py`.

| Decisión | Contenido |
|---|---|
| 1 | Unicidad de `MRUN` en cada tabla |
| 1b | Homologar nombres de columnas entre años |
| 2 | Postulaciones a una fila por estudiante |
| 3 | Panel por año (IP como base, `merge` `left`) |
| 4 | Tipos de datos y no respuesta de ingreso |
| 5 | Egreso reciente y reglas de la brecha |
| 6 | Comparabilidad y concatenación de los 4 años |

In [1]:
# Configuración: rutas, librerías y módulos del proyecto (carpeta src/).
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto (funciona si el notebook se abre desde notebooks/ o
# desde la raíz) para poder importar los módulos de src/.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))

# Los módulos de src/ se importan después de agregar la raíz al path; por
# eso llevan "noqa: E402" (PEP8 pide los imports al inicio del archivo).
from src.carga import (cargar_tablas, comparar_columnas,  # noqa: E402
                       normalizar_columnas)
from src.config import (ANIOS, ARCHIVO_PANEL,  # noqa: E402
                        COLUMNAS_PANEL, ESTADOS_SELECCION, ESTADOS_VALIDOS,
                        TABLAS)
from src.limpieza import (a_formato_largo, armar_panel,  # noqa: E402
                          diferencias_columnas, limpiar_panel,
                          marcar_egreso_reciente, resumir_postulacion,
                          unificar_codigos)

anios = ANIOS
print(f'Raíz del proyecto: {RAIZ}')

Raíz del proyecto: C:\Users\mcped\OneDrive\Desktop\Repositorios\IMT2200 Proyecto Grupal


## 1. Carga de los datos

Los 16 CSV están en una carpeta de Google Drive y se leen **directo desde ahí**
mediante URLs de descarga directa (`src/config.py`). No se guarda ninguna copia
local: pandas descarga cada archivo a memoria y lo lee, así que esta celda tarda
algunos minutos en cada ejecución.

In [2]:
# Lee los 16 CSV directo desde Google Drive (sin guardarlos en el disco).
tablas = cargar_tablas(anios)
ip, sd, ps, m = (tablas[sigla] for sigla in TABLAS)

# Verificación de que las 16 tablas se cargaron (filas x columnas).
for sigla, nombre in TABLAS.items():
    print(f'=== {sigla}: {nombre} ===')
    for a in anios:
        filas, columnas = tablas[sigla][a].shape
        print(f'[OK] {sigla}-{a}: {filas} filas, {columnas} columnas')

Leyendo IP-2026 desde Google Drive ...


Leyendo IP-2025 desde Google Drive ...


Leyendo IP-2024 desde Google Drive ...


Leyendo IP-2023 desde Google Drive ...


Leyendo SD-2026 desde Google Drive ...


Leyendo SD-2025 desde Google Drive ...


... (7 outputs left out)

Leyendo M-2025 desde Google Drive ...


Leyendo M-2024 desde Google Drive ...


Leyendo M-2023 desde Google Drive ...


=== IP: Inscritos y puntajes ===
[OK] IP-2026: 320087 filas, 131 columnas
[OK] IP-2025: 313759 filas, 131 columnas
[OK] IP-2024: 309350 filas, 130 columnas
[OK] IP-2023: 296812 filas, 117 columnas
=== SD: Socioeconómico ===
[OK] SD-2026: 320087 filas, 10 columnas
[OK] SD-2025: 313759 filas, 10 columnas
[OK] SD-2024: 309350 filas, 10 columnas
[OK] SD-2023: 296812 filas, 39 columnas
=== PS: Postulantes y selección ===
[OK] PS-2026: 1694081 filas, 9 columnas
[OK] PS-2025: 1738995 filas, 9 columnas
[OK] PS-2024: 1733891 filas, 9 columnas
[OK] PS-2023: 187226 filas, 84 columnas
=== M: Matrícula ===
[OK] M-2026: 146873 filas, 10 columnas
[OK] M-2025: 140687 filas, 10 columnas
[OK] M-2024: 134492 filas, 10 columnas
[OK] M-2023: 128895 filas, 9 columnas


## 2. Exploración inicial de columnas (proceso 2026)

Revisión de los nombres reales de las columnas antes de tomar decisiones, para no
asumir nombres (por ejemplo, la electiva de Historia se llama `HCSOC`, no `HCSO`).

In [3]:
# Columnas reales de cada tabla (proceso 2026), para no asumir nombres.
for sigla in TABLAS:
    print(f'{sigla}: {tablas[sigla][2026].columns.tolist()}')

IP: ['MRUN', 'ANYO_PROCESO', 'COD_SEXO', 'FECHA_NACIMIENTO', 'RBD', 'CODIGO_ENS', 'LOCAL_EDUCACIONAL', 'UNIDAD_EDUCATIVA', 'NOMBRE_UNIDAD_EDUC', 'RAMA_EDUCACIONAL', 'DEPENDENCIA', 'CODIGO_REGION_EGRESO', 'NOMBRE_REGION_EGRESO', 'CODIGO_PROVINCIA_EGRESO', 'NOMBRE_PROVINCIA_EGRESO', 'CODIGO_COMUNA_EGRESO', 'NOMBRE_COMUNA_EGRESO', 'ANYO_DE_EGRESO', 'PROMEDIO_NOTAS', 'PTJE_NEM', 'PORC_SUP_NOTAS', 'PTJE_RANKING', 'CLEC_REG_ACTUAL', 'MATE1_REG_ACTUAL', 'MATE2_REG_ACTUAL', 'HCSOC_REG_ACTUAL', 'CIEN_REG_ACTUAL', 'CLEC_INV_ACTUAL', 'MATE1_INV_ACTUAL', 'MATE2_INV_ACTUAL', 'HCSOC_INV_ACTUAL', 'CIEN_INV_ACTUAL', 'CLEC_REG_ANTERIOR', 'MATE1_REG_ANTERIOR', 'MATE2_REG_ANTERIOR', 'HCSOC_REG_ANTERIOR', 'CIEN_REG_ANTERIOR', 'CLEC_INV_ANTERIOR', 'MATE1_INV_ANTERIOR', 'MATE2_INV_ANTERIOR', 'HCSOC_INV_ANTERIOR', 'CIEN_INV_ANTERIOR', 'PROMEDIO_CM_MAX', 'CLEC_MAX', 'MATE1_MAX', 'MATE2_MAX', 'HCSOC_MAX', 'CIEN_MAX', 'RINDIO_PROCESO_ANTERIOR', 'RINDIO_PROCESO_ACTUAL', 'HABILITACION_POST', 'PRUEBA_REG_CL', 'FOR

In [4]:
# Columnas clave de IP para la brecha (NEM, ranking, dependencia, PAES).
cols_ip = ip[2026].columns.tolist()
claves = ['NEM', 'RANKING', 'DEPEND', 'CIEN', 'HCSO', 'MATE', 'CLEC',
          'REG_ACTUAL']
print('\n'.join(c for c in cols_ip if any(k in c for k in claves)))

DEPENDENCIA
PTJE_NEM
PTJE_RANKING
CLEC_REG_ACTUAL
MATE1_REG_ACTUAL
MATE2_REG_ACTUAL
HCSOC_REG_ACTUAL
CIEN_REG_ACTUAL
CLEC_INV_ACTUAL
MATE1_INV_ACTUAL
MATE2_INV_ACTUAL
HCSOC_INV_ACTUAL
CIEN_INV_ACTUAL
CLEC_REG_ANTERIOR
MATE1_REG_ANTERIOR
MATE2_REG_ANTERIOR
HCSOC_REG_ANTERIOR
CIEN_REG_ANTERIOR
CLEC_INV_ANTERIOR
MATE1_INV_ANTERIOR
MATE2_INV_ANTERIOR
HCSOC_INV_ANTERIOR
CIEN_INV_ANTERIOR
CLEC_MAX
MATE1_MAX
MATE2_MAX
HCSOC_MAX
CIEN_MAX


In [5]:
# Columnas de año de egreso y rendición, para el filtro de egreso reciente.
claves_egreso = ['EGRE', 'ANIO', 'ANYO', 'PROMOCION', 'RIND']
print([c for c in cols_ip if any(k in c for k in claves_egreso)])

['ANYO_PROCESO', 'CODIGO_REGION_EGRESO', 'NOMBRE_REGION_EGRESO', 'CODIGO_PROVINCIA_EGRESO', 'NOMBRE_PROVINCIA_EGRESO', 'CODIGO_COMUNA_EGRESO', 'NOMBRE_COMUNA_EGRESO', 'ANYO_DE_EGRESO', 'RINDIO_PROCESO_ANTERIOR', 'RINDIO_PROCESO_ACTUAL']


In [6]:
# Distribución de ANYO_DE_EGRESO en el proceso 2026 (5  años más recientes).
print(ip[2026]['ANYO_DE_EGRESO'].value_counts().sort_index().tail(5))

ANYO_DE_EGRESO
2021      5390
2022      8283
2023     15539
2024     54584
2025    220099
Name: count, dtype: int64


## Decisión 1: unicidad de `MRUN` en cada tabla

In [7]:
# Decisión 1: ver si MRUN es único por tabla en el proceso 2026.
tablas_2026 = {'IP': ip[2026], 'SD': sd[2026], 'PS': ps[2026], 'M': m[2026]}
for etiqueta, tabla in tablas_2026.items():
    dup = tabla['MRUN'].duplicated(keep=False).sum()
    print(f'{etiqueta}: {dup} filas repetidas de {len(tabla)}')

IP: 0 filas repetidas de 320087
SD: 0 filas repetidas de 320087
PS: 1685805 filas repetidas de 1694081
M: 0 filas repetidas de 146873


`IP`, `SD` y `M` tienen una fila por estudiante. `PS` tiene una fila por preferencia
(hasta 20 por estudiante), por lo que debe resumirse antes de unir (Decisión 2).

## Decisión 1b: homologar nombres de columnas entre años

Las exploraciones anteriores solo revisaron 2026. Al procesar los otros años se
encontraron diferencias de estructura:

- **PS-2023** viene en formato *ancho* (`ESTADO_PREF_01` … `ESTADO_PREF_20`).
- **IP-2023 y 2024** usan `MATE_*` en algunas columnas de invierno y del proceso
  anterior. No afecta: la brecha usa la rendición regular del proceso actual.
- **M-2023** tiene nombres distintos (`VIA_ADMISION`, `NOMBRE_INSTITUCION`). No
  afecta: de `M` solo se usa `MRUN`.
- **SD-2023** trae columnas adicionales de una encuesta (dispositivos, espacios).

In [8]:
# Decisión 1b: normaliza los nombres de columnas de las 16 tablas (BOM,
# espacios, mayúsculas) y reporta qué columnas de 2026 faltan o sobran.
for sigla, tabla in tablas.items():
    for a in anios:
        normalizar_columnas(tabla[a])
    comparar_columnas(tabla, sigla)

IP-2025: mismas columnas que 2026
IP-2024: faltan 2, sobran 1
    faltan: ['MATE1_INV_ANTERIOR', 'MATE2_INV_ANTERIOR']
    sobran: ['MATE_INV_ANTERIOR']
IP-2023: faltan 21, sobran 7
    faltan: ['CIEN_INV_ANTERIOR', 'CLEC_INV_ANTERIOR', 'CORRECTAS_INV_M1', 'CORRECTAS_INV_M2', 'ERRADAS_INV_M1', 'ERRADAS_INV_M2', 'FORMA_INV_M1', 'FORMA_INV_M2'] ...
    sobran: ['CORRECTAS_INV_MATE', 'ERRADAS_INV_MATE', 'FORMA_INV_MATE', 'MATE_INV_ACTUAL', 'MATE_REG_ANTERIOR', 'OMITIDAS_INV_MATE', 'PRUEBA_INV_MATE']
SD-2025: mismas columnas que 2026
SD-2024: mismas columnas que 2026
SD-2023: faltan 0, sobran 29
    faltan: []
    sobran: ['CONEXION_INSUFICIENTE', 'DISPOSITIVO_1', 'DISPOSITIVO_2', 'DISPOSITIVO_3', 'DISPOSITIVO_4', 'DISPOSITIVO_5', 'ESPACIO_1', 'ESPACIO_2'] ...
PS-2025: mismas columnas que 2026
PS-2024: mismas columnas que 2026
PS-2023: faltan 7, sobran 82
    faltan: ['COD_CARRERA_PREF', 'ESTADO_PREF', 'LUGAR_PREF', 'ORDEN_PREF', 'PTJE_PREF', 'SEXO', 'TIPO_PREF']
    sobran: ['COD_CARRERA_

## Decisión 2: postulaciones a una fila por estudiante

1. Se lleva `PS` a formato largo en los cuatro años (en 2023, las preferencias vacías
   vienen con `ESTADO_PREF = 0` y se eliminan).
2. Se unifica el tipo de `ESTADO_PREF` para que los códigos sean comparables.
3. Se conserva, por estudiante, la preferencia válida mejor ubicada; si no tiene
   ninguna válida, su mejor preferencia inválida.

**Códigos de estado** (libro de códigos de Postulación 2026,
`docs/libro_codigos_postulacion_2026.pdf`, Anexo I): `24` = en lista de
seleccionados, `25` = en lista de espera, `26` = seleccionado en una preferencia
anterior. El resto son causales de postulación no válida (por ejemplo, `36` = no
cumple el promedio mínimo de las pruebas obligatorias).

- **Seleccionado:** tiene alguna preferencia en estado 24 o 26. Se incluye el 26
  porque implica haber quedado seleccionado en una preferencia anterior: en
  PS-2023 hay 179 estudiantes con 26 a los que les falta el registro del 24.
- **Postulación válida:** seleccionado o en lista de espera (24, 25 o 26), lo
  mismo que la "P" de `SITUACION_POSTULANTE`.

In [9]:
# Decisión 2: PS a formato largo, códigos comparables y una fila por MRUN.
ps_largo = {a: unificar_codigos(a_formato_largo(ps[a], a)) for a in anios}

print(f'\nCódigos usados -> seleccionado: {sorted(ESTADOS_SELECCION)}, '
      f'válidos: {sorted(ESTADOS_VALIDOS)}')
print('Códigos de ESTADO_PREF (conteo por año):')
display(pd.concat({a: ps_largo[a]['ESTADO_PREF'].astype('string')
                   .value_counts() for a in anios}, axis=1).fillna(0)
        .astype(int).head(25))

ps_resumido = {a: resumir_postulacion(ps_largo[a]) for a in anios}
for a in anios:
    n_sel = ps_resumido[a]['seleccionado'].sum()
    print(f'PS {a}: {len(ps_largo[a])} filas -> '
          f'{len(ps_resumido[a])} estudiantes ({n_sel} seleccionados)')

PS-2026: formato largo (1694081 filas)
PS-2025: formato largo (1738995 filas)


PS-2024: formato largo (1733891 filas)


PS-2023: formato ancho (187226 filas) -> largo (1523896 filas)

Códigos usados -> seleccionado: [24, 26], válidos: [24, 25, 26]
Códigos de ESTADO_PREF (conteo por año):


,2026,2025,2024,2023
ESTADO_PREF,,,,
26,1012500,1027731,974751,798826
25,387815,426720,475312,494346
24,161907,158218,150492,140558
51,39518,38394,38797,0
36,25156,29989,36318,36190
40,15288,7085,6179,5966
42,11084,10227,11075,6588
31,9394,11778,10688,12273
17,7694,7779,9583,7042


PS 2026: 1694081 filas -> 189399 estudiantes (161907 seleccionados)
PS 2025: 1738995 filas -> 189982 estudiantes (158218 seleccionados)
PS 2024: 1733891 filas -> 188707 estudiantes (150492 seleccionados)
PS 2023: 1523896 filas -> 187226 estudiantes (142902 seleccionados)


**Control de la Decisión 2.** PS-2023 trae la columna `SITUACION_POSTULANTE`
(`P` = tiene al menos una postulación efectiva, en selección o espera; `C` = no
cumple los requisitos de ninguna preferencia). Si la definición de postulación
válida es correcta, `postulacion_valida` debería coincidir con `P`. PS 2024-2026
no traen esta columna, así que el control solo se puede hacer en 2023.

In [10]:
# Control: postulacion_valida vs SITUACION_POSTULANTE = 'P' (solo PS-2023).
control = ps[2023][['MRUN', 'SITUACION_POSTULANTE']].merge(
    ps_resumido[2023][['MRUN', 'postulacion_valida']], on='MRUN',
    how='left')
control['situacion_P'] = (control['SITUACION_POSTULANTE'].astype(str)
                          .str.strip() == 'P')
control['postulacion_valida'] = (control['postulacion_valida']
                                 .fillna(False).astype(bool))
print('Filas: SITUACION_POSTULANTE == P; columnas: postulacion_valida')
display(pd.crosstab(control['situacion_P'], control['postulacion_valida'],
                    margins=True))
coinciden = (control['situacion_P'] == control['postulacion_valida']).mean()
print(f'Coinciden en el {coinciden:.2%} de los {len(control)} postulantes')

Filas: SITUACION_POSTULANTE == P; columnas: postulacion_valida


postulacion_valida,False,True,All
situacion_P,,,
False,7382,0,7382
True,0,179844,179844
All,7382,179844,187226


Coinciden en el 100.00% de los 187226 postulantes


## Decisión 3: panel por año (IP como base, `merge` `left`)

`IP` es el universo completo de inscritos, así que todos los `merge` son `left` para
no perder a quienes se inscribieron pero no postularon, no fueron seleccionados o no
se matricularon: esa comparación es justamente la pregunta de acceso universitario.

In [11]:
# Decisión 3: un panel por año a nivel estudiante (IP + SD + PS + M).
paneles = {a: armar_panel(ip[a], sd[a], ps_resumido[a], m[a], a)
           for a in anios}
print(f'Panel 2026: {len(paneles[2026])} filas, '
      f'{len(paneles[2026].columns)} columnas')

Panel 2026: 320087 filas, 153 columnas


## Decisión 4: tipos de datos y no respuesta de ingreso

- **Tipos:** al leer IP aparece un `DtypeWarning` en `ANYO_DE_EGRESO`, `RBD` y los
  puntajes: hay celdas en blanco y pandas deja la columna completa como texto. Sin
  corregirlo, filtros como `ANYO_DE_EGRESO == 2025` fallan en silencio. Los valores no
  numéricos pasan a `NaN` y se informa cuántos fueron.
- **Ingreso:** el código 99 de `INGRESO_PERCAPITA_GRUPO_FA` es "no responde", no un
  decil (como pidieron los profesores). Se recodifica a `NaN` sin eliminar las filas:
  en el EDA se mantiene como categoría propia, porque quitarlas sesgaría la muestra.

In [12]:
# Decisión 4: columnas clave a numérico y código 99 de ingreso -> NaN.
paneles = {a: limpiar_panel(paneles[a], a) for a in anios}
pct_no_responde = paneles[2026]['INGRESO_PERCAPITA_GRUPO_FA'].isna().mean()
print(f'2026: {pct_no_responde:.1%} de no-respuesta en ingreso percápita')

2026 | RBD: 3200 valores no numéricos -> NaN (ej.: [' '])
2026 | ANYO_DE_EGRESO: 218 valores no numéricos -> NaN (ej.: [' '])


2026 | DEPENDENCIA: 3200 valores no numéricos -> NaN (ej.: [' '])
2026 | CODIGO_ENS: 3200 valores no numéricos -> NaN (ej.: [' '])


2026 | CLEC_REG_ACTUAL: 19496 valores no numéricos -> NaN (ej.: [' '])
2026 | CLEC_REG_ANTERIOR: 252783 valores no numéricos -> NaN (ej.: [' '])


2026 | CLEC_INV_ACTUAL: 288070 valores no numéricos -> NaN (ej.: [' '])
2026 | CLEC_INV_ANTERIOR: 313842 valores no numéricos -> NaN (ej.: [' '])


2026 | MATE1_REG_ACTUAL: 19496 valores no numéricos -> NaN (ej.: [' '])
2026 | MATE1_REG_ANTERIOR: 252783 valores no numéricos -> NaN (ej.: [' '])


2026 | MATE1_INV_ACTUAL: 288070 valores no numéricos -> NaN (ej.: [' '])
2026 | MATE1_INV_ANTERIOR: 313842 valores no numéricos -> NaN (ej.: [' '])


... (33 outputs left out)

2023 | MATE2_REG_ACTUAL: 21598 valores no numéricos -> NaN (ej.: [' '])
2023 | HCSOC_REG_ACTUAL: 21598 valores no numéricos -> NaN (ej.: [' '])


2023 | HCSOC_REG_ANTERIOR: 231069 valores no numéricos -> NaN (ej.: [' '])
2023 | HCSOC_INV_ACTUAL: 263432 valores no numéricos -> NaN (ej.: [' '])


2023 | CIEN_REG_ACTUAL: 21598 valores no numéricos -> NaN (ej.: [' '])
2023 | CIEN_REG_ANTERIOR: 231069 valores no numéricos -> NaN (ej.: [' '])


2023 | CIEN_INV_ACTUAL: 263432 valores no numéricos -> NaN (ej.: [' '])
2026: 27.6% de no-respuesta en ingreso percápita


## Decisión 5: egreso reciente y reglas de la brecha

Reglas confirmadas por el equipo:

- **(a) Primera rendición:** la brecha usa `*_REG_ACTUAL` (y `*_INV_ACTUAL` si fue su
  única rendición), nunca `*_MAX`. En los datos, casi nadie rinde en invierno ni en el
  proceso anterior, así que en la práctica es la rendición regular.
- **(b) Electiva:** `CIEN_REG_ACTUAL` o `HCSOC_REG_ACTUAL`; quienes rinden ambas
  forman un grupo propio.
- **(c) Egreso reciente:** solo estudiantes que egresaron de 4° medio el año en que
  rindieron la PAES. El proceso de admisión *t* se rinde a fines de *t - 1*, por lo
  que se usa `ANYO_DE_EGRESO = ANIO - 1` (`EGRESO_OFFSET = 1`).

Aquí solo se marca el egreso reciente. La brecha se calcula **una sola vez**, en
`src/muestra.py`, dentro de la muestra analítica de `02_eda.ipynb`.

In [13]:
# Decisión 5: marca a los egresados del año de rendición.
paneles = {a: marcar_egreso_reciente(paneles[a], a) for a in anios}
n_reciente = paneles[2026]['egresado_reciente'].sum()
print(f'2026: {n_reciente} de {len(paneles[2026])} son egresados recientes')

2026: 220099 de 320087 son egresados recientes


## Decisión 6: comparabilidad y concatenación de los 4 años

In [14]:
# Decisión 6: revisa diferencias de columnas antes de apilar los 4 años.
# Las diferencias esperadas vienen de la Decisión 1b (columnas no usadas).
for anio, diferencia in diferencias_columnas(paneles).items():
    if diferencia:
        print(f'2023 vs {anio}: {len(diferencia)} columnas distintas '
              f'(ej.: {sorted(diferencia)[:5]})')

panel_completo = pd.concat(paneles.values(), ignore_index=True)
print(f'Panel completo: {len(panel_completo)} filas, '
      f'{panel_completo["ANIO"].nunique()} años')

2023 vs 2026: 61 columnas distintas (ej.: ['CIEN_INV_ANTERIOR', 'CLEC_INV_ANTERIOR', 'CONEXION_INSUFICIENTE', 'CORRECTAS_INV_M1', 'CORRECTAS_INV_M2'])
2023 vs 2025: 61 columnas distintas (ej.: ['CIEN_INV_ANTERIOR', 'CLEC_INV_ANTERIOR', 'CONEXION_INSUFICIENTE', 'CORRECTAS_INV_M1', 'CORRECTAS_INV_M2'])
2023 vs 2024: 60 columnas distintas (ej.: ['CIEN_INV_ANTERIOR', 'CLEC_INV_ANTERIOR', 'CONEXION_INSUFICIENTE', 'CORRECTAS_INV_M1', 'CORRECTAS_INV_M2'])


Panel completo: 1240008 filas, 4 años


## Guardar el panel

Se guardan solo las columnas que usa el EDA, en formato Parquet (mucho más liviano
y rápido que CSV, y conserva los tipos de datos).

In [15]:
# Guarda el panel para 02_eda.ipynb (data/processed/ está en .gitignore).
columnas = [c for c in COLUMNAS_PANEL if c in panel_completo.columns]
ARCHIVO_PANEL.parent.mkdir(parents=True, exist_ok=True)
panel_completo[columnas].to_parquet(ARCHIVO_PANEL, index=False)
print(f'Guardado {ARCHIVO_PANEL.name}: {len(panel_completo)} filas, '
      f'{len(columnas)} columnas')

Guardado panel_completo.parquet: 1240008 filas, 37 columnas
